# 🩺 Predicción de Riesgo de Enfermedad Cardiovascular
**Autor:** Jorge Sánchez Hernández  
**Proyecto:** Machine Learning supervisado  
**Datos:** Encuesta de Salud España 2025 *(Ministerio de Sanidad)*

# Introducción y objetivo

- ¿Qué problema de salud pública abordas? --> Riesgo cardiovascular
- ¿Qué variable quieres predecir? --> Enfermedad Cardiovascular ECV
- ¿Por qué es importante minimizar los falsos negativos (priorizar **Recall**)? --> No podemos decirle a un paciente enfermo que esta sano. 

## 1. Carga de datos

In [3]:
# Librerías utilizadas
import pandas as pd
import numpy as np
import pyreadstat

In [4]:
'''Creamos un diccionario con los nombres de las variables. Esta información esta disponible 
en un archivo excel descargable en la web del Ministerio de Sanidad'''

nombres = {
    # Variable objetivo (luego se combinan en 'ecv' --> enfermedad cardiovascular)
    "C5a_2": "infarto", "C5a_3": "angina_y_otras_coronarias",
    "C5a_4": "otra_cardiopatia", "C5a_24": "ictus",

    # Peso muestral (solo lo utilizaremos en el EDA, no es un predictor)
    "FACTORADULTO": "factor",

    # Variables demograficas y socioeconómicas
    "EDADa": "edad", "SEXOa": "sexo", "NIVEST": "nivel_estudios",
    "CLASE_PR": "clase_social", "CCAA": "ccaa",

    # Cálculo del IMC
    "N1": "altura_cm", "N2": "peso_kg", "IMC": "imc_categoria",

    # Antecedentes clínicos que son factores de riesgo cv
    "C5a_1": "tension_alta", "C5a_15": "colesterol_alto", "C5a_12": "diabetes",
    "C5a_11": "epoc", "C5a_30": "problemas_rinon", "C5a_25": "migrana",
    "C5a_27": "tumor_maligno",

    # Salud mental de los pacientes
    "C5a_21": "depresion", "C5a_22": "ansiedad", "C5a_23": "otros_problemas_mentales",
    "SEVERIDAD_DEPRESIVA": "severidad_depresiva",

    # Tabaco (factor de riesgo)
    "Q1": "fuma", "Q2": "cigarrillos_dia", "Q6": "humo_ajeno", "Q8": "vapeo",

    # Alcohol (factor de riesgo)
    "R1": "frec_alcohol", "R3": "frec_atracon_alcohol",

    # Ejercicio y sedentarismo (para el cálculo de nueva variable 'horas_sentado')
    "O1": "actividad_trabajo", "O2": "ejercicio_ocio", "O3": "dias_camina",
    "O7": "dias_deporte", "O9": "dias_fuerza",
    "O10_1": "sentado_h", "O10_2": "sentado_min",

    # Dieta
    "P1_1": "dieta_fruta", "P1_9": "dieta_verdura", "P1_10": "dieta_legumbres",
    "P1_6": "dieta_pescado", "P1_2": "dieta_carne_procesada", "P1_12": "dieta_dulces",
    "P1_13": "dieta_refrescos", "P1_14": "dieta_comida_rapida", "P1_15": "dieta_aperitivos",

    # Apoyo social de los pacientes
    "S1": "apoyo_personas",
}

In [ ]:

# PROXY_0 son las personas que han contestado a la encuesta por si mismos
# se carga para filtrar después, pero no se renombra
columnas = ["PROXY_0"] + list(nombres.keys())

# Dos lecturas del mismo archivo (mismas filas y mismo orden) 
df_codigo = pd.read_spss("../data/raw/ESdEadulto_2025.sav", usecols=columnas, convert_categoricals=False)  # para el modelo
df_etiquetas = pd.read_spss("../data/raw/ESdEadulto_2025.sav", usecols=columnas)                           # para ver y graficar

# En df_codigo los códigos vienen como string y los pasamos a número
df_codigo = df_codigo.apply(pd.to_numeric, errors="coerce")

# Renombrar las columnas
df_codigo = df_codigo.rename(columns=nombres)
df_etiquetas = df_etiquetas.rename(columns=nombres)

# Comprobaciones
print(df_codigo.shape, df_etiquetas.shape)        # (21641, 48) las dos
print(df_codigo["PROXY_0"].value_counts())        # 1 -> 20357, 2 -> 1284
print(df_etiquetas["PROXY_0"].value_counts())     # Sí / No
print(df_codigo["sexo"].unique(), df_etiquetas["sexo"].unique())   # [1 2] y Hombre/Mujer


# Guardamos la selección en data/processed
df_codigo.to_csv("../data/processed/ESdEadulto_2025_codigos.csv", index=False)
df_etiquetas.to_csv("../data/processed/ESdEadulto_2025_etiquetas.csv", index=False)

(21641, 48) (21641, 48)
PROXY_0
1    20357
2     1284
Name: count, dtype: int64
PROXY_0
Sí    20357
No     1284
Name: count, dtype: int64
[1 2] ['Hombre', 'Mujer']
Categories (2, str): ['Hombre', 'Mujer']
